<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 24


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать базовый класс **Notification** в C#, который будет представлять уведомления
пользователям. На основе этого класса разработать 2-3 производных класса,
демонстрирующих принципы наследования и полиморфизма. В каждом из классов
должны быть реализованы новые атрибуты и методы, а также переопределены
некоторые методы базового класса для демонстрации полиморфизма.
Требования к базовому классу Notification:

• Атрибуты: ID уведомления (NotificationId), Текст уведомления (MessageText), Тип уведомления (Type).

• **Методы**:
- DisplayNotification(): метод для отображения уведомления
пользователю.
- SendNotification(): метод для отправки уведомления.
- GetNotificationDetails(): метод для получения деталей уведомления.
Требования к производным классам:
1. EmailУведомление (EmailNotification): Должно содержать дополнительные
атрибуты, такие как Адрес электронной почты (EmailAddress).
Метод SendNotification() должен быть переопределен для отправки
уведомления по электронной почте.
2. SMSУведомление (SMSNotification): Должно содержать дополнительные
атрибуты, такие как Номер телефона (PhoneNumber).
Метод SendNotification() должен быть переопределен для отправки
уведомления через SMS.
3. PushУведомление (PushNotification) (если требуется третий класс): Должно
содержать дополнительные атрибуты, такие как Платформа (Platform,
например, iOS или Android). Метод DisplayNotification() должен быть
переопределен для отображения уведомления на мобильной платформе.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
using System;
using System.Collections.Generic;

// --- ИНТЕРФЕЙСЫ ---

public interface INotificationSender
{
    void Send();
    string GetStatus();
}

public interface IPrioritizable
{
    int Priority { get; set; }
    bool IsHighPriority();
}

// --- БАЗОВЫЙ КЛАСС ---

public class Notification : INotificationSender
{
    private int _notificationId;
    private string _messageText;
    private string _type;
    private DateTime _createdDate;
    private bool _isRead;

    private string _sender;
    private string _recipient;
    private List<string> _tags = new List<string>();

    // Новые атрибуты
    private bool _isArchived;
    private DateTime? _sentDate;
    private string _category;

    public int NotificationId { get => _notificationId; set => _notificationId = value; }
    public string MessageText
    {
        get => _messageText;
        set
        {
            if (string.IsNullOrEmpty(value))
                throw new ArgumentException("Сообщение не может быть пустым!");
            _messageText = value;
        }
    }
    public string Type { get => _type; protected set => _type = value; }
    public DateTime CreatedDate { get => _createdDate; private set => _createdDate = value; }
    public bool IsRead { get => _isRead; set => _isRead = value; }
    public string Sender { get => _sender; set => _sender = value; }
    public string Recipient { get => _recipient; set => _recipient = value; }
    public List<string> Tags { get => _tags; }

    // Новые свойства
    public bool IsArchived { get => _isArchived; set => _isArchived = value; }
    public DateTime? SentDate { get => _sentDate; set => _sentDate = value; }
    public string Category { get => _category; set => _category = value; }

    public Notification(int notificationId, string sender, string recipient, string messageText, string type)
    {
        NotificationId = notificationId;
        Sender = sender;
        Recipient = recipient;
        MessageText = messageText;
        Type = type;
        CreatedDate = DateTime.Now;
        IsRead = false;
    }

    public void AddTag(string tag)
    {
        if (!string.IsNullOrWhiteSpace(tag))
        {
            _tags.Add(tag.ToLower());
            Console.WriteLine($"К уведомлению {NotificationId} добавлен тег: '{tag.ToLower()}'.");
        }
    }

    public void MarkAsRead()
    {
        IsRead = true;
        Console.WriteLine($"Уведомление {NotificationId} отмечено как прочитанное.");
    }

    public virtual void Send()
    {
        SentDate = DateTime.Now;
        OnSent(EventArgs.Empty);
        Console.WriteLine($"[Отправка] Уведомление #{NotificationId} для {Recipient} от {Sender}.");
    }

    public virtual void Send(DateTime scheduledTime)
    {
        Console.WriteLine($"[Отложенная отправка] Уведомление #{NotificationId} будет отправлено {scheduledTime} для {Recipient}.");
    }

    public virtual string GetStatus()
    {
        return $"Статус: {(IsRead ? "Прочитано" : "Не прочитано")}, Создано: {CreatedDate}, Отправлено: {(SentDate.HasValue ? SentDate.Value.ToString() : "Не отправлено")}";
    }

    public virtual void DisplayInfo()
    {
        Console.WriteLine($"ID: {NotificationId}, Тип: {Type}, От: {Sender}, Кому: {Recipient}");
        Console.WriteLine($"Категория: {Category ?? "Без категории"}");
        Console.WriteLine($"Сообщение: \"{MessageText}\"");
    }

    // Делегат и событие для отправки уведомления
    public delegate void NotificationSentEventHandler(object sender, EventArgs e);
    public event NotificationSentEventHandler NotificationSent;

    protected virtual void OnSent(EventArgs e)
    {
        NotificationSent?.Invoke(this, e);
    }
}

// --- ПРОИЗВОДНЫЕ КЛАССЫ ---

public class EmailNotification : Notification, IPrioritizable
{
    public string Subject { get; set; }
    public List<string> Attachments { get; private set; } = new List<string>();
    public int Priority { get; set; }

    // Новые атрибуты
    public bool IsEncrypted { get; set; }
    public string ReplyTo { get; set; }

    public EmailNotification(int id, string sender, string recipientEmail, string subject, string message)
        : base(id, sender, recipientEmail, message, "Email")
    {
        Subject = subject;
        Priority = 5;
        Category = "Письмо";
    }

    public void AddAttachment(string fileName)
    {
        Attachments.Add(fileName);
        Console.WriteLine($"К письму '{Subject}' добавлен файл: {fileName}");
    }

    public bool IsHighPriority() => Priority > 8;

    public override void Send()
    {
        base.Send();
        Console.WriteLine($"[Отправка Email] Кому: {Recipient}, Тема: {Subject}");
        if (Attachments.Count > 0)
            Console.WriteLine($"С {Attachments.Count} вложениями.");
        if (IsEncrypted)
            Console.WriteLine("Письмо зашифровано.");
        if (IsHighPriority())
            Console.WriteLine("ВАЖНОЕ ПИСЬМО!");
    }
}

public class SmsNotification : Notification
{
    public string Operator { get; private set; }
    private bool _deliveryReportRequired;

    // Новые свойства
    public int SmsLength { get; private set; }
    public bool IsUnicode { get; set; }
    public string CountryCode { get; set; }

    public SmsNotification(int id, string sender, string recipientPhone, string message, string mobileOperator)
        : base(id, sender, recipientPhone, message, "SMS")
    {
        Operator = mobileOperator;
        _deliveryReportRequired = false;
        SmsLength = message.Length;
        IsUnicode = false;
    }

    public void RequestDeliveryReport()
    {
        _deliveryReportRequired = true;
        Console.WriteLine("Запрошен отчет о доставке SMS.");
    }

    public override void Send()
    {
        base.Send();
        Console.WriteLine($"[Отправка SMS] На номер {Recipient} через оператора {Operator}.");
        Console.WriteLine($"Длина SMS: {SmsLength} символов, Юникод: {IsUnicode}");
        if (_deliveryReportRequired)
            Console.WriteLine("Будет запрошен статус доставки.");
    }
}

public class PushNotification : Notification, IPrioritizable
{
    public string Platform { get; set; }
    public string Sound { get; set; }
    public int Priority { get; set; }

    // Новые свойства
    public bool IsSilent { get; set; }
    public int BadgeNumber { get; set; }

    public PushNotification(int id, string sender, string recipientDeviceId, string message, string platform)
        : base(id, sender, recipientDeviceId, message, "Push")
    {
        Platform = platform;
        Sound = "default";
        Priority = 5;
        Category = "Push-уведомление";
    }

    public bool IsHighPriority() => Priority > 7;

    public override void Send()
    {
        base.Send();
        Console.WriteLine($"[Отправка Push] На платформу {Platform} (устройство: {Recipient}).");
        Console.WriteLine($"Звук: {Sound}, Тихий режим: {(IsSilent ? "Да" : "Нет")}, Приоритет: {Priority}, Значок: {BadgeNumber}");
    }

    public override void Send(DateTime scheduledTime)
    {
        Console.WriteLine($"[Отложенный Push] Уведомление для {Platform} будет отправлено в {scheduledTime}.");
    }
}

// --- GENERIC КЛАСС ---

public class NotificationQueue<T> where T : INotificationSender
{
    private Queue<T> _queue = new Queue<T>();

    public int Count => _queue.Count;

    // Событие для уведомления о добавлении в очередь
    public event EventHandler<T> NotificationEnqueued;

    public void Enqueue(T notification)
    {
        Console.WriteLine($"Уведомление типа {notification.GetType().Name} добавлено в очередь.");
        _queue.Enqueue(notification);
        NotificationEnqueued?.Invoke(this, notification);
    }

    public void ProcessNext()
    {
        if (_queue.Count > 0)
        {
            T notification = _queue.Dequeue();
            Console.WriteLine($"--- Обработка уведомления из очереди ---");
            notification.Send();
            Console.WriteLine("-------------------------------------");
        }
        else
        {
            Console.WriteLine("Очередь уведомлений пуста.");
        }
    }
}

// --- ДЕМОНСТРАЦИЯ ---

var email = new EmailNotification(101, "system@corp.com", "user@example.com", "Квартальный отчет", "Отчет во вложении.");
email.AddTag("work");
email.AddTag("reports");
email.AddAttachment("report_q3.pdf");
email.Priority = 9;
email.IsEncrypted = true;
email.ReplyTo = "reply@corp.com";

var sms = new SmsNotification(102, "BANK", "+79991234567", "Код подтверждения: 5566", "MTS");
sms.AddTag("security");
sms.RequestDeliveryReport();
sms.IsUnicode = true;
sms.CountryCode = "+7";

var push = new PushNotification(103, "app.games", "device-token-xyz", "Ваша энергия восстановлена!", "Android");
push.Sound = "notification.mp3";
push.Priority = 8;
push.IsSilent = false;
push.BadgeNumber = 5;

Console.WriteLine("\n=== Демонстрация полиморфизма и событий ===\n");

Notification[] notifications = { email, sms, push };

foreach (var n in notifications)
{
    n.DisplayInfo();
    n.Send();
    n.Send(DateTime.Now.AddMinutes(30));
    Console.WriteLine(n.GetStatus());
    Console.WriteLine("---");
}

Console.WriteLine("\n=== Демонстрация Generic класса NotificationQueue с событием ===\n");

var notificationProcessor = new NotificationQueue<Notification>();

notificationProcessor.NotificationEnqueued += (sender, notification) =>
{
    Console.WriteLine($"Событие: уведомление типа {notification.GetType().Name} добавлено в очередь.");
};

notificationProcessor.Enqueue(email);
notificationProcessor.Enqueue(sms);
notificationProcessor.Enqueue(push);

Console.WriteLine($"\nВ очереди {notificationProcessor.Count} уведомления. Начинаем обработку...");

notificationProcessor.ProcessNext();
notificationProcessor.ProcessNext();
notificationProcessor.ProcessNext();
notificationProcessor.ProcessNext(); // Попытка обработать пустую очередь


К уведомлению 101 добавлен тег: 'work'.
К уведомлению 101 добавлен тег: 'reports'.
К письму 'Квартальный отчет' добавлен файл: report_q3.pdf
К уведомлению 102 добавлен тег: 'security'.
Запрошен отчет о доставке SMS.

=== Демонстрация полиморфизма и событий ===

ID: 101, Тип: Email, От: system@corp.com, Кому: user@example.com
Категория: Письмо
Сообщение: "Отчет во вложении."
[Отправка] Уведомление #101 для user@example.com от system@corp.com.
[Отправка Email] Кому: user@example.com, Тема: Квартальный отчет
С 1 вложениями.
Письмо зашифровано.
ВАЖНОЕ ПИСЬМО!
[Отложенная отправка] Уведомление #101 будет отправлено 11/15/2025 6:34:04 PM для user@example.com.
Статус: Не прочитано, Создано: 11/15/2025 6:04:04 PM, Отправлено: 11/15/2025 6:04:04 PM
---
ID: 102, Тип: SMS, От: BANK, Кому: +79991234567
Категория: Без категории
Сообщение: "Код подтверждения: 5566"
[Отправка] Уведомление #102 для +79991234567 от BANK.
[Отправка SMS] На номер +79991234567 через оператора MTS.
Длина SMS: 23 символов, 